# Frutalândia — Image-to-Video de verdade (R$ 0)

Este notebook usa **a imagem original do elenco versionada no GitHub**, não os bonecos de esferas do Blender. Gera primeiro **uma prévia de 2 segundos** pela rede de difusão LTX-Video (pesos abertos). Os personagens podem sofrer deriva visual entre quadros; valide a amostra antes de gerar as seis cenas.

**Requisitos:** GPU NVIDIA atribuída pelo Colab, internet para baixar modelos livres, sessão Colab ativa. Não usa serviços pagos, API keys, Kaggle, publicação automática ou login de terceiros além do Colab.

**Não promete GPU gratuita:** se o Colab não conceder GPU, a execução falha com aviso explícito.

In [ ]:
#@title 1. Selecionar cena para teste
CENA = 's01' #@param ['s01','s02','s03','s04','s05','s06']
MODO = 'preview' #@param ['preview','render-scene']
APROVO_PRODUZIR_8_SEGUNDOS = False #@param {type:'boolean'}
PASSOS = 28 #@param {type:'integer'}
BAIXAR_VIDEO = True #@param {type:'boolean'}

import os,subprocess,sys,pathlib,json
PROJECT=pathlib.Path('/content/moneyPrintTest')
OUT=pathlib.Path('/content/frutalandia-i2v')
OUT.mkdir(parents=True,exist_ok=True)
if MODO=='render-scene' and not APROVO_PRODUZIR_8_SEGUNDOS:
    raise RuntimeError('Antes de produzir 8 segundos, aprove o preview explicitamente.')
print('Cena:',CENA,'Modo:',MODO)

In [ ]:
#@title 2. Abrir repositório com a imagem-mestre incluída
REPO='https://github.com/nutricionistaalmeidavh-spec/moneyPrintTest.git'
BRANCH='feat/frutalandia-i2v-referencia-fixa'
if not (PROJECT/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,REPO,str(PROJECT)],check=True,timeout=180)
else:
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only','origin',BRANCH],check=True,timeout=180)
sys.path.insert(0,str(PROJECT))
from IPython.display import display
from PIL import Image
img=Image.open(PROJECT/'assets/frutalandia/elenco-master.webp')
print('Referência visual:',img.size)
display(img.resize((960,540)))

In [ ]:
#@title 3. Ativar GPU e instalar pipeline open weights
import torch,shutil
from scripts.frutalandia_i2v import check_available_gpu
check_available_gpu(torch)
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','pillow>=10,<13','diffusers>=0.35,<1','transformers>=4.48,<5','accelerate>=1,<2','safetensors>=0.4','imageio-ffmpeg>=0.5','av>=14'],check=True,timeout=420)
if not shutil.which('ffprobe'):
    subprocess.run(['apt-get','update','-qq'],check=True,timeout=180)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg'],check=True,timeout=240)
print('Infraestrutura gratuita preparada, GPU:',torch.cuda.get_device_name(0))

In [ ]:
#@title 4. Preparar frames por personagem e conferir identidade
os.chdir(PROJECT)
subprocess.run([sys.executable,'-m','scripts.frutalandia_i2v','prepare','--out',str(OUT)],check=True,timeout=90)
from PIL import Image
frame=Image.open(OUT/'first-frames'/('first-frame-'+CENA+'.png'))
display(frame)
print('A imagem é apenas referência, AINDA NÃO é um vídeo gerado.')

In [ ]:
#@title 5. Gerar preview real do modelo (pode exigir muitos GB e GPU)
# Este processo exige uma GPU NVIDIA de verdade, não finge gerar frames com zoom ou loop.
cmd=[sys.executable,'-u','-m','scripts.frutalandia_i2v',MODO,'--scene',CENA,'--out',str(OUT),'--steps',str(PASSOS)]
if MODO=='render-scene': cmd.append('--approve-full')
env=os.environ.copy()
env['PYTHONUNBUFFERED']='1'
env['HF_HUB_DISABLE_TELEMETRY']='1'
print('INFERÊNCIA REAL iniciada com imagem de referência. Download inicial dos pesos pode ser grande.')
subprocess.run(cmd,cwd=PROJECT,env=env,check=True,timeout=14400)
print('Inferência terminou e FFprobe validou o MP4.')

In [ ]:
#@title 6. Exibir e baixar vídeo verdadeiro
from IPython.display import Video,display
from google.colab import files
stem='preview-' if MODO=='preview' else 'scene-'
clip=OUT/(stem+CENA+'.mp4')
assert clip.is_file() and clip.stat().st_size>10000,'Vídeo não gerado — confira erros acima'
from scripts.frutalandia_i2v import validate_preview
print(validate_preview(clip,2 if MODO=='preview' else 8))
display(Video(str(clip),embed=True,html_attributes='controls'))
if BAIXAR_VIDEO: files.download(str(clip))

## Limites e custos

- **Núcleo R$0 / aberto:** Diffusers + LTX-Video (pesos disponíveis publicamente), FFmpeg/Pillow. Consulte a licença dos pesos antes de exploração comercial.
- **Colab gratuito:** a conta pode não receber GPU ou pode desconectar. Colab é somente uma opção para executar o núcleo, não dependência de produção.
- **Não há sincronismo labial nem sotaque nordestino no vídeo de prévia:** a animação é visual. Piper é a opção de TTS do episódio completo, que requer aprovação visual prévia.
- **Qualidade:** a imagem de referência pode sofrer alterações pelos modelos de difusão; um preview técnico não equivale a aprovação editorial. O workflow não publica em redes nem faz merge.